In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path

import importlib
import track

# Development helper: uncomment after editing track.py.
# importlib.reload(track)

# Load Snapshot

按区域窗口加载单日 OFES NP30 快照。MOM3 Arakawa B-grid 的 u/v 均由四角共置到示踪物中心；参照 JAMSTEC 对公开 OFES DODS 的通用错误 mbar 警告，结合交付层位将 lev 按深度米解释，w 保留在层界面。

In [ ]:
snap = track.load_ofes_snapshot(
    '2003-04-05', lon_bounds=(142, 150), lat_bounds=(32, 39),
    depth_bounds=(0, 1100),
)
print(list(snap.keys()))
print(f"do2: {snap['do2'].shape}, u: {snap['u'].shape}, w: {snap['w'].shape}")

# Quick-Look Horizontal Slice

In [ ]:
track.plot_ofes_snapshot_quick(snap, variable='do2', depth=600.0)

# Vertical Profile Extraction

从快照中双线性插值提取定深虚拟剖面；Depth / do2 / temp / salinity 继续送入通用单剖面 detector。

In [ ]:
prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                         variables=['do2', 'temp', 'salinity'])
prof.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True)
for ax, var, label in zip(axes, ['do2', 'temp', 'salinity'],
                          ['DO₂ (μmol kg⁻¹)', 'Potential temp (°C)', 'Salinity (PSS-78)']):
    ax.plot(prof[var], prof['Depth'], linewidth=1.2)
    ax.set_xlabel(label)
    ax.invert_yaxis()
axes[0].set_ylabel('Depth (m)')
fig.suptitle(f"OFES profile  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
fig.tight_layout()

# δDO Detection on OFES

复用观测 pipeline 的 `calculate_delta_do` 在 OFES 虚拟剖面上检测异常。

In [ ]:
det_cfg = track.make_detection_config('do')

result = track.detect_ofes_delta_do(snap, lon=144.5, lat=35.0,
                                    detection_config=det_cfg)
result

In [ ]:
det_prof = track.extract_ofes_profile_interp(snap, lon=144.5, lat=35.0,
                                             variables=['do2'])
fig, ax = plt.subplots(figsize=(5, 7))
ax.plot(det_prof['do2'], det_prof['Depth'], 'b-', linewidth=1.2)
ax.axhline(det_cfg.anomaly_min_depth, color='gray', ls='--', alpha=0.5,
           label=f'min depth {det_cfg.anomaly_min_depth:.0f} m')
for _, row in result.iterrows():
    ax.plot(row['do_value'], row['depth'], 'ro', ms=7)
    ax.annotate(f"ΔDO={row['delta_do']:.1f}", (row['do_value'], row['depth']),
                textcoords='offset points', xytext=(8, 0), color='red', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('DO₂ (μmol kg⁻¹)')
ax.set_ylabel('Depth (m)')
ax.set_title(f"δDO detection  144.5°E, 35°N  {snap['date'].strftime('%Y-%m-%d')}")
ax.legend(loc='lower left')
fig.tight_layout()

# Annual Event Catalog

年度 ΔDO20/35/50 扫描已经通过 parity 与掩码审计。这里仅读取固定运行，不在 notebook 中重扫或重排候选。

In [ ]:
catalog_run = Path(
    'plot_outputs/do/ofes_np30_ke/ofes_delta_do_catalog/'
    '20030101_20031231_cf957935d38a'
)
catalog_manifest = json.loads((catalog_run / 'manifest.json').read_text())
if (
    catalog_manifest.get('schema_version') != 2
    or catalog_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed depth/m schema-v2 OFES catalog.')
event_catalog = pd.read_parquet(catalog_run / 'event_catalog.parquet')
{
    'status': catalog_manifest['status'],
    'days': catalog_manifest['completed_days'],
    'events': len(event_catalog),
}

# Ranked Water-Mass Diagnostics

正式候选固定为质量排名前五的 DO50 events。等密度水团、heave、动力量与负对照均从已验证诊断表读取，500–900 m 排名只作敏感性分析。

In [ ]:
diagnostic_run = catalog_run / (
    'event_diagnostics/ofes_events_21efbe902ab7'
)
diagnostic_manifest = json.loads((diagnostic_run / 'manifest.json').read_text())
if (
    diagnostic_manifest.get('schema_version') != 2
    or diagnostic_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use completed diagnostics from the schema-v2 catalog.')
selected_events = pd.read_parquet(
    diagnostic_run / 'selected_events.parquet'
)
event_diagnostics = pd.read_parquet(
    diagnostic_run / 'event_diagnostic_summary.parquet'
)
event_diagnostics

# Deep-Event Population Diagnostics

将已预声明的 59 个 DO50、500–900 m 事件扩展为峰值日总体诊断，连续比较 water-mass/heave、深层旋转/strain，并以同点、异常面积等效半径核心加权和 50 km 上界三种口径检查表层 SSH/涡度表达；不启动聚类或轨迹。

In [ ]:
population_run = diagnostic_run / (
    'event_population/ofes_population_254ae68988a6'
)
population_manifest = json.loads((population_run / 'manifest.json').read_text())
if (
    population_manifest.get('schema_version') != 2
    or population_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES population run.')
population_summary = json.loads(
    (population_run / 'population_summary.json').read_text()
)
population_summary

# Regime-Stratified Onset Diagnostics

从 59-event population 中按预声明排序和可诊断性门槛客观选取总体第一、应变主导第一、表层核心弱或反号的旋转主导第一。比较 detector start 前五天、首次进入 500 m 以下和 peak；水平锋生仅含解析水平速度梯度项，不等同于已经证明下沉或 subduction。

In [ ]:
onset_run = population_run / (
    'event_onset/ofes_onset_cf012b5274b0'
)
onset_manifest = json.loads((onset_run / 'manifest.json').read_text())
if (
    onset_manifest.get('schema_version') != 2
    or onset_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES onset run.')
onset_summary = pd.read_parquet(
    onset_run / 'event_onset_summary.parquet'
)
onset_summary

# Event-Flow Consistency Audit

以相邻 detected objects 的质心位移作为 Eulerian 结构移动，以两端最大 ΔDO 核心的固定深度 u/v 梯形平均作为平流代理。方向一致只说明 linked-object motion 与解析流相容；这不是粒子轨迹，且核心与质心偏移被单独报告。

In [ ]:
flow_run = onset_run / (
    'event_flow/ofes_flow_2b5ac7cf4911'
)
flow_manifest = json.loads((flow_run / 'manifest.json').read_text())
if (
    flow_manifest.get('schema_version') != 2
    or flow_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES event-flow audit.')
flow_summary = pd.read_parquet(
    flow_run / 'event_flow_summary.parquet'
)
flow_summary

# Water-Mass Patch Continuity

在三个预注册事件中追踪同密度面高 DO、同号低 spice patch，并分别审计连续日数、平移后掩膜 IoU、质心位移与直接质心流速。patch gate 只控制来源路径措辞，不阻止经过数值验证的二维轨迹运行。

In [ ]:
patch_run = onset_run / (
    'event_patch/ofes_patch_2146a27fee9d'
)
patch_manifest = json.loads((patch_run / 'manifest.json').read_text())
if patch_manifest.get('status') != 'complete':
    raise RuntimeError('Use a completed OFES patch-continuity run.')
patch_summary = pd.read_parquet(patch_run / 'patch_summary.parquet')
patch_summary

# Event Lifecycle Diagnostics

把 population 峰值日的同一 rotation/strain 和表层表达判据推广到 56 个严格事件的全部 observed days；peak parity 必须精确继承 26/29 同极性和 7/29 表层弱或反转。SCV-compatible 只作为预注册动力代理，并保留应变和锋生对照。

In [ ]:
lifecycle_run = population_run / (
    'event_lifecycle/ofes_lifecycle_f7290df019c2'
)
lifecycle_manifest = json.loads((lifecycle_run / 'manifest.json').read_text())
if (
    lifecycle_manifest.get('schema_version') != 2
    or lifecycle_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES lifecycle run.')
lifecycle_summary = pd.read_parquet(
    lifecycle_run / 'lifecycle_event_summary.parquet'
)
lifecycle_summary

# Event Evolution Diagnostics

读取已验证的 schema-v2 演化运行：每个固定候选包含 ±10 天语境、start/peak/end 等密度面图，并共享年度总览、负对照和 Hosoda 三日期集成检验。如需重建，调用 `track.build_ofes_event_evolution_diagnostics`。

In [ ]:
evolution_run = diagnostic_run / (
    'event_evolution/ofes_evolution_46b47184fd21'
)
evolution_manifest = json.loads((evolution_run / 'manifest.json').read_text())
if (
    evolution_manifest.get('schema_version') != 2
    or evolution_manifest.get('status') != 'complete'
):
    raise RuntimeError('Use a completed schema-v2 OFES evolution run.')
evolution_manifest['outputs']['figures']

# Fixed-Depth Trajectory Ensembles

读取已通过解析平移、固体旋转、1800/3600/7200 s 步长敏感性及 1/3/5-day 可逆性验证的二维固定深度 ensemble。peak backward、observed-start forward 和纯数值 replay 分开报告；该运行作为三维 u/v/w 路径的独立水平对照。

In [ ]:
trajectory_run = onset_run / (
    'trajectory_2d/ofes_trajectory2d_db507a8add65'
)
trajectory_manifest = json.loads((trajectory_run / 'manifest.json').read_text())
if (
    trajectory_manifest.get('status') != 'complete'
    or not trajectory_manifest.get('all_validation_passed')
):
    raise RuntimeError('Use a validated OFES fixed-depth trajectory run.')
trajectory_summary = pd.read_parquet(trajectory_run / 'event_summary.parquet')
trajectory_summary

# Three-Dimensional Trajectory Ensembles

读取已通过 B-grid 连续方程 w 门控、三维解析场、步长和可逆性验证的 u/v/w ensemble。同起点固定深度对照与 release-ring/初始层敏感性分开报告；E000002 的 resolved downward-pathway 门槛失败，不解锁 material-source 或严格 SCV 措辞。

In [ ]:
trajectory_3d_run = onset_run / (
    'trajectory_3d/ofes_trajectory3d_b7a76a0347ad'
)
trajectory_3d_manifest = json.loads(
    (trajectory_3d_run / 'manifest.json').read_text()
)
if (
    trajectory_3d_manifest.get('status') != 'complete'
    or not trajectory_3d_manifest.get('all_validation_passed')
):
    raise RuntimeError('Use a validated OFES three-dimensional trajectory run.')
trajectory_3d_summary = pd.read_parquet(
    trajectory_3d_run / 'event_summary.parquet'
)
trajectory_3d_summary